# 3 — Leverage, risk and liquidation

**Question.** A levered long's gross return is mechanically leverage × the spot
return, so "4× perp ≈ 4× spot, leverage wins" is true *by construction* in an
up-window and says little. The real questions: on a **risk-adjusted** basis does
leverage add anything, and what happens **unconditionally**, across all regimes?

In [1]:
import warnings

import pandas as pd

from perp_spot import backtest, config, data, metrics

warnings.simplefilter("ignore")
market = data.load_market(config.PRIMARY_SYMBOL)

## 3.1 In a single bull window, Sharpe is ~unchanged; drawdown is not

Leverage scales the numerator (return) **and** the denominator (vol) of the
Sharpe ratio together — so risk-adjusted return is roughly invariant, while
**max drawdown scales with leverage**. There is no free risk-adjusted lunch.

In [2]:
books = backtest.run_books(market, config.CASE_START, config.CASE_END)
spot, perp = books["spot"], books["perp"]
print(f"Window {config.CASE_START} → {config.CASE_END}")
print(f"Spot 1x : return {metrics.summarize(spot.equity)['total_return']:+.1%}  "
      f"Sharpe {metrics.sharpe(metrics.to_returns(spot.equity)):.2f}  "
      f"maxDD {metrics.max_drawdown(spot.equity):.1%}")
print(f"Perp 4x : return {metrics.summarize(perp.equity)['total_return']:+.1%}  "
      f"Sharpe {metrics.sharpe(metrics.to_returns(perp.equity)):.2f}  "
      f"maxDD {metrics.max_drawdown(perp.equity):.1%}")
print(f"\nBreak-even funding APR (4x perp net = spot): "
      f"{backtest.break_even_funding(market, config.CASE_START, config.CASE_END):.0%}  "
      f"— i.e. how expensive funding must get before leverage stops paying here.")

Window 2024-01-01 → 2024-03-20
Spot 1x : return +60.2%  Sharpe 3.78  maxDD -15.8%
Perp 4x : return +215.2%  Sharpe 3.85  maxDD -50.5%

Break-even funding APR (4x perp net = spot): 170%  — i.e. how expensive funding must get before leverage stops paying here.


## 3.2 The path endpoint-only P&L throws away: liquidation in a bear regime

Endpoint-only P&L (first Open, last Close) cannot see a margin wipe mid-window.
Marking to market daily with a maintenance-margin guard, the 4× long is
**liquidated** in the 2022 sell-off while the carry is untouched.

In [3]:
bear = backtest.run_books(market, "2022-04-01", "2022-06-30")
for res in bear.values():
    flag = f"  LIQUIDATED {pd.Timestamp(res.liquidation_date).date()}" if res.liquidated else ""
    print(f"{res.name:18} return {res.final_equity / config.DEFAULT_INVESTMENT - 1:+7.1%}{flag}")
print("\nSee docs/figures/equity_bear.png for the equity paths.")

Spot 1x            return  -56.4%
Perp 4x            return -100.0%  LIQUIDATED 2022-05-08
Carry (Δ-neutral)  return   +0.5%

See docs/figures/equity_bear.png for the equity paths.


## 3.3 Unconditionally, 4× is a coin-flip that blows up ~1 quarter in 5

Across all 16 non-overlapping quarters of 2021–2024:

In [4]:
wf = backtest.walk_forward(market, freq="QE")
summary = wf.groupby("strategy").agg(
    median_return=("total_return", "median"),
    worst_return=("total_return", "min"),
    median_sharpe=("sharpe", "median"),
    worst_drawdown=("max_drawdown", "min"),
    liquidations=("liquidated", "sum"),
    n=("total_return", "size"),
)
summary.round(3)

,median_return,worst_return,median_sharpe,worst_drawdown,liquidations,n
strategy,,,,,,
Carry (Δ-neutral),0.022,0.004,9.089,-0.005,0,16
Perp 4x,-0.031,-1.000,1.083,-1.000,3,16
Spot 1x,0.030,-0.564,0.475,-0.594,0,16


The 4× perp's **median** quarterly return is negative and it is liquidated in
several quarters — judging it from one bull window would be survivorship bias.
The carry's distribution is tight and positive; spot is the honest middle.

## 3.4 Does the story generalize? BTC vs ETH vs SOL

Re-run the walk-forward per asset and compare median quarterly returns and
liquidation counts.

In [5]:
rows = []
for sym in config.SYMBOLS:
    m = data.load_market(sym)
    w = backtest.walk_forward(m, freq="QE")
    g = w.groupby("strategy").agg(med_ret=("total_return", "median"),
                                  liq=("liquidated", "sum"),
                                  med_sharpe=("sharpe", "median"))
    g["symbol"] = sym
    rows.append(g.reset_index())
multi = pd.concat(rows).pivot_table(index="symbol", columns="strategy",
                                    values="med_ret").round(3)
print("Median quarterly return by symbol × strategy:")
print(multi.to_string())
liqs = pd.concat(rows).pivot_table(index="symbol", columns="strategy",
                                   values="liq", aggfunc="sum")
print("\nLiquidation counts (4x perp) by symbol:")
print(liqs.to_string())

Median quarterly return by symbol × strategy:
strategy  Carry (Δ-neutral)  Perp 4x  Spot 1x
symbol                                       
BTCUSDT               0.022   -0.031    0.030
ETHUSDT               0.025    0.465    0.203
SOLUSDT               0.014    0.312    0.166

Liquidation counts (4x perp) by symbol:
strategy  Carry (Δ-neutral)  Perp 4x  Spot 1x
symbol                                       
BTCUSDT                   0        3        0
ETHUSDT                   0        3        0
SOLUSDT                   0        6        0


### Takeaways
1. Leverage buys **no risk-adjusted edge** (Sharpe ~invariant) while scaling
   drawdown — and at 4× it produces outright ruin in bear regimes.
2. The "perp beats spot" conclusion is a single-window, single-asset artifact;
   it does not survive a walk-forward across regimes or across BTC/ETH/SOL.
3. The defensible portfolio statement: *the edge here is the funding carry, not
   the leverage* — harvested delta-neutrally, it is the only book with an
   attractive risk-adjusted profile across all regimes.